# Hierarchical Clustering on Well Log Data
## Machine learning for energy applications — Module 3, Exercise 4

In this exercise, we will apply **Hierarchical (Agglomerative) Clustering** to well log data consisting of two measurements: **DTC** (compressional sonic slowness) and **RHOB** (bulk density). These two logs are commonly used in petrophysics to distinguish lithologies and fluid types.

Hierarchical clustering builds a tree of clusters (a **dendrogram**) by iteratively merging the closest clusters. Unlike K-Means, it does not require specifying the number of clusters upfront — instead, we can visually inspect the dendrogram and choose a cut height.

**Topics covered:**
1. Load and explore the dataset
2. Standardize features
3. Compute the linkage matrix (Ward method)
4. Plot and interpret the dendrogram
5. Apply AgglomerativeClustering
6. Visualize clusters
7. Compare linkage methods
8. Compare with K-Means
9. Summary

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['figure.dpi'] = 100

---
## 1. Load and Explore the Dataset

Load `hierarchical_clustering_dataset.csv` and inspect its shape, statistics, and create a scatter plot of DTC vs RHOB.

In [ ]:
# Load the dataset
df = pd.read_csv('hierarchical_clustering_dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nSummary statistics:")
df.describe()

In [ ]:
# Scatter plot of the raw data
plt.figure(figsize=(8, 6))
plt.scatter(df['DTC'], df['RHOB'], alpha=0.5, s=20, edgecolors='k', linewidth=0.3)
plt.xlabel('DTC (Compressional Sonic Slowness)')
plt.ylabel('RHOB (Bulk Density)')
plt.title('Well Log Data — DTC vs RHOB')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of each feature
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['DTC'], bins=40, color='steelblue', edgecolor='black')
axes[0].set_xlabel('DTC')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of DTC')

axes[1].hist(df['RHOB'], bins=40, color='darkorange', edgecolor='black')
axes[1].set_xlabel('RHOB')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of RHOB')

plt.tight_layout()
plt.show()

---
## 2. Standardize Features

Since DTC and RHOB have very different scales, we standardize before clustering.

**Exploratory analysis of one fixed dataset.** The scaler is fitted on all 82 samples because we only describe these data; nothing is predicted for new samples, so there is no training/test split.

In [ ]:
# Standardize
scaler = StandardScaler()
X = df[['DTC', 'RHOB']].values
X_scaled = scaler.fit_transform(X)

print(f"Shape: {X_scaled.shape}")
print(f"Mean after scaling: {X_scaled.mean(axis=0).round(4)}")
print(f"Std after scaling:  {X_scaled.std(axis=0).round(4)}")

---
## 3. Compute the Linkage Matrix (Ward Method)

The linkage matrix encodes the hierarchical merging process. We use the **Ward** method, which minimizes the total within-cluster variance at each merge step.

In [ ]:
# Compute the Ward linkage matrix
Z = linkage(X_scaled, method='ward')

print(f"Linkage matrix shape: {Z.shape}")
print(f"\nFirst 5 merges:")
print("  [cluster_i, cluster_j, distance, n_samples_in_new_cluster]")
for row in Z[:5]:
    print(f"  {row}")

---
## 4. Plot the Dendrogram

The dendrogram visualizes the hierarchical merging. We can choose a **cut height** to determine the number of clusters. Choose the cut inside a large gap between successive merge heights: a long vertical branch means that groups stayed separate over a wide range of heights. Leaf order is not a physical coordinate.

In [ ]:
# Full dendrogram (truncated for readability)
plt.figure(figsize=(14, 7))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90,
           leaf_font_size=10, show_contracted=True)
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage)')
plt.xlabel('Cluster Size or Sample Index')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

In [ ]:
# Dendrogram with a horizontal cut line
# Choose the cut inside the largest gap between successive merge heights
last_heights = Z[-10:, 2]
print("Last 10 merge heights:", last_heights.round(2))
print("Gaps between them:    ", np.diff(last_heights).round(2))
cut_height = 10.0  # inside the largest gap (between the last two merges -> 2 clusters)

plt.figure(figsize=(14, 7))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90,
           leaf_font_size=10, show_contracted=True)
plt.axhline(y=cut_height, color='red', linestyle='--', linewidth=2,
            label=f'Cut height = {cut_height}')
plt.title('Dendrogram with Cut Height')
plt.xlabel('Cluster Size or Sample Index')
plt.ylabel('Distance')
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

# Determine number of clusters from cut height
labels_cut = fcluster(Z, t=cut_height, criterion='distance')
n_clusters_from_cut = len(np.unique(labels_cut))
print(f"Number of clusters at cut height {cut_height}: {n_clusters_from_cut}")

---
## 5. Apply AgglomerativeClustering

Use scikit-learn's `AgglomerativeClustering` with the chosen number of clusters.

In [ ]:
# Apply AgglomerativeClustering
n_clusters = n_clusters_from_cut
agg = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
labels_agg = agg.fit_predict(X_scaled)

print(f"Number of clusters: {n_clusters}")
print(f"\nCluster sizes:")
unique, counts = np.unique(labels_agg, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Cluster {u}: {c} samples")

sil = silhouette_score(X_scaled, labels_agg)
print(f"\nSilhouette Score: {sil:.4f}")

---
## 6. Visualize Clusters

Plot the clustering results in both the original feature space and the standardized space.

In [ ]:
# Visualize clusters in original feature space
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Original space
scatter1 = axes[0].scatter(df['DTC'], df['RHOB'], c=labels_agg, cmap='viridis',
                           alpha=0.6, s=20, edgecolors='k', linewidth=0.3)
plt.colorbar(scatter1, ax=axes[0], label='Cluster')
axes[0].set_xlabel('DTC')
axes[0].set_ylabel('RHOB')
axes[0].set_title(f'Hierarchical Clustering (Ward, K={n_clusters}) — Original Space')

# Standardized space
scatter2 = axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_agg, cmap='viridis',
                           alpha=0.6, s=20, edgecolors='k', linewidth=0.3)
plt.colorbar(scatter2, ax=axes[1], label='Cluster')
axes[1].set_xlabel('DTC (standardized)')
axes[1].set_ylabel('RHOB (standardized)')
axes[1].set_title(f'Hierarchical Clustering (Ward, K={n_clusters}) — Standardized Space')

plt.tight_layout()
plt.show()

---
## 7. Compare Linkage Methods

Compare the four main linkage methods: **Ward**, **Complete**, **Average**, and **Single**.

In [ ]:
# Compare linkage methods
linkage_methods = ['ward', 'complete', 'average', 'single']

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
results_linkage = []

for ax, method in zip(axes.ravel(), linkage_methods):
    agg_m = AgglomerativeClustering(n_clusters=n_clusters, linkage=method)
    labels_m = agg_m.fit_predict(X_scaled)
    sil_m = silhouette_score(X_scaled, labels_m)
    
    results_linkage.append({'Method': method, 'Silhouette': sil_m})
    
    scatter = ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_m, cmap='viridis',
                         alpha=0.6, s=20, edgecolors='k', linewidth=0.3)
    ax.set_title(f'{method.capitalize()} Linkage — Silhouette: {sil_m:.4f}')
    ax.set_xlabel('DTC (standardized)')
    ax.set_ylabel('RHOB (standardized)')

plt.suptitle(f'Linkage Method Comparison (K={n_clusters})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

results_linkage_df = pd.DataFrame(results_linkage)
print("\nLinkage Method Comparison:")
results_linkage_df

In [ ]:
# Dendrograms for each linkage method
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, method in zip(axes.ravel(), linkage_methods):
    Z_m = linkage(X_scaled, method=method)
    dendrogram(Z_m, truncate_mode='lastp', p=20, ax=ax, leaf_rotation=90,
               leaf_font_size=8, show_contracted=True)
    ax.set_title(f'{method.capitalize()} Linkage Dendrogram')
    ax.set_xlabel('Cluster / Sample')
    ax.set_ylabel('Distance')

plt.suptitle('Dendrogram Comparison by Linkage Method', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Compare with K-Means

Apply K-Means with the same number of clusters and compare the results side-by-side with hierarchical clustering.

In [ ]:
# K-Means for comparison
km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels_km = km.fit_predict(X_scaled)

sil_km = silhouette_score(X_scaled, labels_km)
sil_agg = silhouette_score(X_scaled, labels_agg)

print(f"Silhouette Score — Hierarchical (Ward): {sil_agg:.4f}")
print(f"Silhouette Score — K-Means:             {sil_km:.4f}")

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Hierarchical
scatter1 = axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_agg, cmap='viridis',
                           alpha=0.6, s=20, edgecolors='k', linewidth=0.3)
axes[0].set_title(f'Hierarchical (Ward) — Silhouette: {sil_agg:.4f}')
axes[0].set_xlabel('DTC (standardized)')
axes[0].set_ylabel('RHOB (standardized)')

# K-Means
scatter2 = axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_km, cmap='viridis',
                           alpha=0.6, s=20, edgecolors='k', linewidth=0.3)
# Plot K-Means centroids
axes[1].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                c='red', marker='X', s=200, edgecolors='black', linewidth=2, label='Centroids')
axes[1].set_title(f'K-Means — Silhouette: {sil_km:.4f}')
axes[1].set_xlabel('DTC (standardized)')
axes[1].set_ylabel('RHOB (standardized)')
axes[1].legend()

plt.suptitle(f'Hierarchical vs K-Means (K={n_clusters})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Summary

In this exercise we:

1. **Loaded and explored** a well log dataset with DTC (sonic slowness) and RHOB (bulk density) measurements.
2. **Standardized** the features to ensure equal weighting in distance calculations.
3. **Computed the Ward linkage matrix** and plotted a **dendrogram** to visualize the hierarchical merging process.
4. **Chose a cut height** in the largest gap between merge heights to determine the number of clusters.
5. **Applied AgglomerativeClustering** and visualized the resulting clusters in both original and standardized feature spaces.
6. **Compared four linkage methods** (Ward, Complete, Average, Single) using silhouette scores and visual inspection.
7. **Compared hierarchical clustering with K-Means** on the same data and number of clusters.

**Key takeaways:**
- Hierarchical clustering does not require specifying K upfront — cut the dendrogram inside a large gap between successive merge heights.
- The **Ward** method tends to produce compact, evenly-sized clusters (similar to K-Means).
- **Single** linkage can produce elongated "chain" clusters, while **Complete** linkage tends to produce more compact clusters.
- For well log data, both hierarchical clustering and K-Means propose groupings of the log responses; which lithologies or fluids they correspond to, if any, is a hypothesis to test against core, cuttings or other logs.